# Model Building and Training
This notebook contains model training and evaluation for both the e-commerce and credit card fraud datasets.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, f1_score, average_precision_score, precision_recall_curve, roc_auc_score)
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load data
ecom_df = pd.read_csv('../data/raw/Fraud_Data.csv')
credit_df = pd.read_csv('../data/raw/creditcard.csv')

In [ ]:
# --- E-commerce Data Preparation ---
df = ecom_df.copy()
# Feature engineering (if already done, skip or load processed)
df['signup_time'] = pd.to_datetime(df['signup_time'])
df['purchase_time'] = pd.to_datetime(df['purchase_time'])
df['transaction_count'] = df.groupby('user_id')['user_id'].transform('count')
df['prev_purchase_time'] = df.groupby('user_id')['purchase_time'].shift(1)
df['time_since_last_purchase'] = (df['purchase_time'] - df['prev_purchase_time']).dt.total_seconds() / 3600
df['time_since_last_purchase'].fillna(0, inplace=True)
df['hour_of_day'] = df['purchase_time'].dt.hour
df['day_of_week'] = df['purchase_time'].dt.dayofweek
df['time_since_signup'] = (df['purchase_time'] - df['signup_time']).dt.total_seconds() / 3600
df.drop(columns=['prev_purchase_time'], inplace=True)
# Prepare features
X = df.drop(columns=['class', 'purchase_time', 'signup_time', 'user_id', 'ip_address'])
y = df['class']
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)

In [ ]:
# --- Credit Card Data Preparation ---
X_credit = credit_df.drop(columns=['Class'])
y_credit = credit_df['Class']
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X_credit, y_credit, stratify=y_credit, test_size=0.3, random_state=42)

In [ ]:
# --- Model Training & Evaluation Function ---
def evaluate_model(model, X_test, y_test, model_name="Model"):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    cm = confusion_matrix(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc_pr = average_precision_score(y_test, y_proba) if y_proba is not None else None
    auc_roc = roc_auc_score(y_test, y_proba) if y_proba is not None else None
    print(f"{model_name} F1-score: {f1:.4f}")
    print(f"{model_name} AUC-PR: {auc_pr:.4f}")
    print(f"{model_name} AUC-ROC: {auc_roc:.4f}")
    print(f"{model_name} Confusion Matrix:\n{cm}")
    if y_proba is not None:
        precision, recall, _ = precision_recall_curve(y_test, y_proba)
        plt.figure(figsize=(5,4))
        plt.plot(recall, precision, label=f'AUC-PR={auc_pr:.3f}')
        plt.xlabel('Recall')
        plt.ylabel('Precision')
        plt.title(f'Precision-Recall Curve ({model_name})')
        plt.legend()
        plt.show()

## E-commerce Dataset: Model Training and Evaluation

In [ ]:
# Logistic Regression
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train, y_train)
evaluate_model(lr, X_test, y_test, model_name="Logistic Regression (Ecom)")
# Random Forest
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)
evaluate_model(rf, X_test, y_test, model_name="Random Forest (Ecom)")

## Credit Card Dataset: Model Training and Evaluation

In [ ]:
# Logistic Regression
lr_c = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_c.fit(Xc_train, yc_train)
evaluate_model(lr_c, Xc_test, yc_test, model_name="Logistic Regression (Credit)")
# Random Forest
rf_c = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_c.fit(Xc_train, yc_train)
evaluate_model(rf_c, Xc_test, yc_test, model_name="Random Forest (Credit)")

### Model Comparison and Justification

- Compare models using F1-score and AUC-PR, which are robust to class imbalance.
- The best model is the one with the highest AUC-PR and F1-score, as these reflect both precision and recall for the minority (fraud) class.
- Random Forest is expected to outperform Logistic Regression due to its ability to capture nonlinearities and interactions, but results will be interpreted based on the actual metrics above.